# Stage 2 — Faithfulness Results

This notebook analyses the faithfulness results from Stage 2 expert steering experiments.

**Three datasets**:
- **Counterfactual** (`FaithEval-Counterfactual`): MCQ where the context contradicts training knowledge. A faithful model follows the context. Metric: accuracy on the context-supported answer.
- **Unanswerable** (`FaithEval-Unanswerable`): Questions the context cannot answer. A faithful model abstains. Metric: proportion of correct abstentions.
- **SQuAD control** (`MCTest`): Standard QA where context and knowledge agree. Should be unaffected by steering. Metric: accuracy.

**Steering direction**: faithfulness negative — suppress confabulation-preferred experts.

**Conditions**: `baseline` (no steering) vs `hard` (full expert deactivation).

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font='serif')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})

RESULTS_PATH = '/scratch/sc23jc3/stage2_results/results.json'
OUT_DIR = 'figures'
os.makedirs(OUT_DIR, exist_ok=True)

with open(RESULTS_PATH) as f:
    results = json.load(f)

faith = results['faithfulness']
print('Faithfulness datasets:', list(faith.keys()))

## 1. Accuracy: Baseline vs Hard Steering

In [ ]:
dataset_labels = {
    'counterfactual': 'Counterfactual\n(FaithEval)',
    'unanswerable':   'Unanswerable\n(FaithEval)',
    'mctest':         'SQuAD Control\n(MCTest)',
}

present = [(k, v) for k, v in dataset_labels.items() if k in faith]
labels    = [v for _, v in present]
baselines = [faith[k].get('baseline', float('nan')) for k, _ in present]
hards     = [faith[k].get('hard',     float('nan')) for k, _ in present]
deltas    = [h - b for b, h in zip(baselines, hards)]

x = np.arange(len(labels))
w = 0.3

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w/2, baselines, width=w, label='Baseline',      color='#7f8c8d', alpha=0.85)
ax.bar(x + w/2, hards,     width=w, label='Hard steering', color='#27ae60', alpha=0.85)

for i, (b, h, d) in enumerate(zip(baselines, hards, deltas)):
    if not (np.isnan(b) or np.isnan(h)):
        sign = '+' if d >= 0 else ''
        ax.annotate(f'{sign}{d:.3f}', xy=(x[i] + w/2, h), xytext=(0, 4),
                    textcoords='offset points', ha='center', fontsize=10,
                    color='#27ae60', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
ax.set_title('Faithfulness Results: Baseline vs Hard Expert Deactivation')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'faithfulness_results.png'), dpi=300, bbox_inches='tight')
plt.show()

## 2. Numerical Summary

In [ ]:
print(f"{'Dataset':30s} {'Baseline':>10} {'Hard':>10} {'Delta':>10}")
print('-' * 65)
for k, label in dataset_labels.items():
    if k not in faith:
        continue
    b = faith[k].get('baseline', float('nan'))
    h = faith[k].get('hard',     float('nan'))
    d = h - b if not (np.isnan(b) or np.isnan(h)) else float('nan')
    print(f"{label.replace(chr(10), ' '):30s} {b:10.3f} {h:10.3f} {d:+10.3f}")

## Discussion

*(Fill in after examining the results.)*

**Counterfactual**:
- If hard steering raises accuracy, the model is following context more faithfully after suppressing confabulation experts. Strong causal finding.

**Unanswerable**:
- If hard steering raises abstention accuracy, the model is becoming more appropriately agnostic — not hallucinating answers when the context is insufficient.

**SQuAD control**:
- This should ideally remain flat. A significant drop would indicate steering degrades general reading comprehension — a sign the intervention is too crude rather than specific.
- A small drop is acceptable; a large drop undermines the causal interpretation.

**Key question**: does faithfulness improve on the test datasets *without* degrading the control?